In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
os.chdir('/Users/saemhabeeb/Desktop/SEM7/COL774/ass/ass4/COL774-A4-Maze-Dataset')

In [3]:
df = pd.read_csv('train_6x6_mazes.csv')
df_test = pd.read_csv('test_6x6_mazes.csv')
df.head()

,input_sequence,output_path,maze_type
0,"['<ADJLIST_START>', '(3,5)', '<-->', '(2,5)', ...","['(0,3)', '(0,4)', '(1,4)', '(1,5)', '(2,5)', ...",forkless
1,"['<ADJLIST_START>', '(5,4)', '<-->', '(5,3)', ...","['(3,0)', '(2,0)', '(2,1)', '(2,2)', '(1,2)', ...",forkless
2,"['<ADJLIST_START>', '(5,3)', '<-->', '(4,3)', ...","['(2,2)', '(3,2)', '(3,3)', '(3,4)', '(4,4)', ...",forked
3,"['<ADJLIST_START>', '(2,0)', '<-->', '(1,0)', ...","['(5,0)', '(5,1)', '(5,2)', '(4,2)', '(4,1)', ...",forked
4,"['<ADJLIST_START>', '(3,0)', '<-->', '(3,1)', ...","['(0,0)', '(1,0)', '(2,0)', '(3,0)', '(3,1)', ...",forkless


In [4]:
df.shape

(80000, 3)

In [5]:
type(df['input_sequence'].iloc[0])

str

In [6]:
print((df.iloc[0]['input_sequence'])[0])

[


In [7]:
inp_tokens = eval(df['input_sequence'].iloc[0])
out_tokens = eval(df['output_path'].iloc[0])

print(inp_tokens+out_tokens)

['<ADJLIST_START>', '(3,5)', '<-->', '(2,5)', ';', '(3,2)', '<-->', '(3,3)', ';', '(3,4)', '<-->', '(3,3)', ';', '(0,4)', '<-->', '(1,4)', ';', '(2,3)', '<-->', '(2,4)', ';', '(1,5)', '<-->', '(2,5)', ';', '(3,4)', '<-->', '(3,5)', ';', '(0,1)', '<-->', '(0,0)', ';', '(0,2)', '<-->', '(0,1)', ';', '(3,2)', '<-->', '(2,2)', ';', '(2,3)', '<-->', '(1,3)', ';', '(1,4)', '<-->', '(1,5)', ';', '(1,0)', '<-->', '(0,0)', ';', '(1,3)', '<-->', '(1,2)', ';', '(1,2)', '<-->', '(2,2)', ';', '(0,4)', '<-->', '(0,3)', ';', '(0,2)', '<-->', '(0,3)', ';', '<ADJLIST_END>', '<ORIGIN_START>', '(0,3)', '<ORIGIN_END>', '<TARGET_START>', '(1,3)', '<TARGET_END>', '<PATH_START>', '(0,3)', '(0,4)', '(1,4)', '(1,5)', '(2,5)', '(3,5)', '(3,4)', '(3,3)', '(3,2)', '(2,2)', '(1,2)', '(1,3)', '<PATH_END>']


In [8]:
def parse_coords(s):
    nums = re.findall(r"-?\d+", s)
    return tuple(map(int, nums)) if len(nums) == 2 else None

def extract_between(tag, text):
    """Accepts many tag styles: <TAG_START>, <TAG START>, <TAG-START>, <TAGSTART>, etc."""
    patterns = [
        rf"<\s*{tag}\s*[_\-\s]?\s*START\s*>(.*?)<\s*{tag}\s*[_\-\s]?\s*END\s*>",
        rf"<\s*{tag}START\s*>(.*?)<\s*{tag}END\s*>",
        rf"<\s*{tag}\s*START\s*>(.*?)<\s*{tag}\s*END\s*>",
        rf"<\s*{tag.replace(' ', '_')}\s*START\s*>(.*?)<\s*{tag.replace(' ', '_')}\s*END\s*>",
    ]
    for p in patterns:
        m = re.search(p, text, re.S | re.I)
        if m:
            return m.group(1).strip()
    raise ValueError(f"Could not find section for tag '{tag}'. Tried multiple patterns.")

def plot_maze(tokens):
    text = " ".join(tokens)
    adj_section = extract_between("ADJLIST", text)
    origin_section = extract_between("ORIGIN", text)
    target_section = extract_between("TARGET", text)
    path_section = extract_between("PATH", text)

    origin = parse_coords(origin_section)
    target = parse_coords(target_section)

    # parse edges like "(r,c) <--> (r2,c2)"
    edge_matches = re.findall(r"\(\s*-?\d+\s*,\s*-?\d+\s*\)\s*<-->\s*\(\s*-?\d+\s*,\s*-?\d+\s*\)", adj_section)
    edges = []
    for em in edge_matches:
        coords = re.findall(r"\(\s*-?\d+\s*,\s*-?\d+\s*\)", em)
        a = parse_coords(coords[0])
        b = parse_coords(coords[1])
        edges.append((a, b))

    # parse path coordinates (supports parenthesized coords)
    path = [parse_coords(p) for p in re.findall(r"\(\s*-?\d+\s*,\s*-?\d+\s*\)", path_section)]
    if not path:
        # fallback: "r,c" tokens without parentheses
        nums = re.findall(r"-?\d+\s*,\s*-?\d+", path_section)
        path = [tuple(map(int, re.findall(r"-?\d+", s))) for s in nums]

    if not edges:
        raise ValueError("No edges found in adjacency list. Ensure format '(r,c) <--> (r2,c2)'.")

    # --------------------------
    # Grid size (cells indexed with (0,0) = top-left)
    # --------------------------
    all_nodes = {n for e in edges for n in e if n is not None}
    all_nodes.update([origin, target])
    all_nodes.update([p for p in path if p is not None])
    rows = 6
    cols = 6


    vertical_walls = np.ones((rows, cols + 1), dtype=bool)
    horizontal_walls = np.ones((rows + 1, cols), dtype=bool)

    for (r1, c1), (r2, c2) in edges:
        if r1 == r2:
            # same row, adjacent columns -> remove vertical wall between them
            c_between = min(c1, c2) + 1  # column index of the vertical segment between c and c+1
            vertical_walls[r1, c_between] = False
        elif c1 == c2:
            # same column, adjacent rows -> remove horizontal wall between them
            r_between = min(r1, r2) + 1  # row index of horizontal segment between r and r+1
            horizontal_walls[r_between, c1] = False
        else:
            # diagonal or invalid — ignore, but warn
            print(f"Warning: non-grid edge {(r1,c1)} <--> {(r2,c2)} ignored")


    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_aspect('equal')

    # Draw a full light-gray grid (every cell border)
    for r in range(rows):
        for c in range(cols):
            x0, x1 = c, c + 1
            y_top = rows - r
            y_bot = rows - r - 1
            ax.plot([x0, x1], [y_top, y_top], color='lightgray', lw=2)   # top
            ax.plot([x0, x1], [y_bot, y_bot], color='lightgray', lw=2)   # bottom
            ax.plot([x0, x0], [y_bot, y_top], color='lightgray', lw=2)   # left
            ax.plot([x1, x1], [y_bot, y_top], color='lightgray', lw=2)   # right

    # Draw vertical walls (black) using vertical_walls[r,c]
    for r in range(rows):
        for c in range(cols + 1):
            if vertical_walls[r, c]:
                x = c
                y_top = rows - r
                y_bot = rows - r - 1
                ax.plot([x, x], [y_bot, y_top], color='black', lw=5, solid_capstyle='butt')

    # Draw horizontal walls (black)
    for r in range(rows + 1):
        for c in range(cols):
            if horizontal_walls[r, c]:
                y = rows - r
                ax.plot([c, c + 1], [y, y], color='black', lw=5, solid_capstyle='butt')

    shade_path_cells = True
    if shade_path_cells and path:
        for (r, c) in path:
            # rectangle corners in plot coords
            x0, x1 = c, c + 1
            y_top = rows - r
            y_bot = rows - r - 1
            rect = plt.Rectangle((x0, y_bot), 1, 1, facecolor=(1, 0.9, 0.9), edgecolor=None, zorder=0)
            ax.add_patch(rect)

    # Plot path line and markers (convert (r,c) top-left -> matplotlib coords)
    if path:
        path_x = [c + 0.5 for (r, c) in path]
        path_y = [rows - r - 0.5 for (r, c) in path]
        ax.plot(path_x, path_y, linestyle='--', linewidth=2, color='red', zorder=4)
        ax.scatter(path_x[0], path_y[0], c='red', s=80, marker='o', zorder=5)  # start
        ax.scatter(path_x[-1], path_y[-1], c='red', s=80, marker='x', zorder=5)  # goal
    else:
        # if no path, still mark origin/target
        ox, oy = origin[1] + 0.5, rows - origin[0] - 0.5
        tx, ty = target[1] + 0.5, rows - target[0] - 0.5
        ax.scatter(ox, oy, c='red', s=80, marker='o', zorder=5)
        ax.scatter(tx, ty, c='red', s=80, marker='x', zorder=5)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    plt.yticks([]) 
    ax.set_xlabel("col")
    ax.set_ylabel("row")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_maze(inp_tokens+out_tokens)

NameError: name 'plot_maze' is not defined

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

In [11]:
device = torch.device(0)

In [12]:
class BahdanauAttention(nn.Module):
    def __init__(self, n_enc, n_dec, n_attn):
        super().__init__()
        self.W_a = nn.Linear(n_dec, n_attn, bias=False)
        self.U_a = nn.Linear(n_enc, n_attn, bias=False)
        self.v_a = nn.Linear(n_attn, 1, bias=False)

    def forward(self, h, s_i_1,mask):
        """
        h:(batch, seq_len, n)
        s_{i-1}: (batch, n)  -> prev hidden state of decoder
        """
        energy = torch.tanh(self.U_a(h) + self.W_a(s_i_1).unsqueeze(1))

        alpha = self.v_a(energy).squeeze(-1)
        if mask is not None:
            alpha = alpha.masked_fill(~mask, -1e4)
        attn_weights = F.softmax(alpha, dim=-1)

        # context: (batch, hidden_size)
        context = torch.bmm(attn_weights.unsqueeze(1), h).squeeze(1)

        return context, attn_weights

In [13]:
class RNNenc(nn.Module):
    def __init__(self, K_x, m_x, n_enc, padding_idx):
        super().__init__()
        self.embeder = nn.Embedding(K_x, m_x, padding_idx=padding_idx)
        self.rnn = nn.RNN(
            input_size=m_x,
            hidden_size=n_enc,
            num_layers=2,
            nonlinearity='tanh',
            batch_first=True
        )
    def forward(self, x):
        emb = self.embeder(x)
        h, h_last = self.rnn(emb)
        return h, h_last


In [ ]:
class RNNdec(nn.Module):
    def __init__(self, K_y, m_y, n_dec, n_enc, n_attn,out_dim, paddin_idx):
        super().__init__()
        self.embeder = nn.Embedding(K_y, m_y,padding_idx=paddin_idx)
        self.attention = BahdanauAttention(n_enc,n_dec=n_dec, n_attn=n_attn)
        self.rnn = nn.RNN(m_y+n_enc, 
                          n_dec,
                          num_layers=2,
                          batch_first=True,
                          nonlinearity='tanh',)
        self.out = nn.Linear(n_dec+m_y+n_attn, out_dim)
    def forward(self, h, h_last, y=None, T_y=None, 
                teacher_forcing = 0.5, mask=None,starter=1):
        batch_size = h.size(0)
        teacher_size = int(batch_size*teacher_forcing)
        device = h.device
        if y is not None: T_y = y.size(1)-1
        out_len = T_y
        s_t = h_last
        y_in = torch.full((batch_size,),starter, device=device)
        outputs = []
        all_attn_weights = []
        for t in range(out_len):
            emb = self.embeder(y_in).unsqueeze(1)
            context, attn_weights = self.attention.forward(h, s_t[-1], mask=mask)
            context_unsq = context.unsqueeze(1) # so that rnn runs for 1 unit
            rnn_in = torch.cat([emb, context_unsq], dim=2)
            all_attn_weights.append(attn_weights.unsqueeze(1)) # unsqueeze so we can cat all later on
            _, s_t = self.rnn(rnn_in, s_t)
            out_in = torch.cat([s_t[-1],context,emb.squeeze(1)], dim=1)
            logits = self.out(out_in) # (batch,out_dim)
            outputs.append(logits.unsqueeze(1))
            if t < out_len-1:
                y_in = logits.argmax(dim=1)
                if y is not None:
                    # teacher forcing
                    idxs = np.arange(batch_size)
                    np.random.shuffle(idxs)
                    y_in[idxs[:teacher_size]] = y[idxs[:teacher_size],t+1]
        outputs = torch.cat(outputs, dim=1)
        all_attn_weights = torch.cat(all_attn_weights,dim=1)
        return outputs, all_attn_weights

In [65]:
class seq2seqBahdanau(nn.Module):
    def __init__(self, K_x,m_x, K_y,m_y, n_enc, n_attn, n_dec, out_dim):
        super().__init__()
        self.encoder = RNNenc(K_x, m_x, n_enc, padding_idx=0)
        self.decoder = RNNdec(K_y, m_y, n_dec, n_enc, n_attn, out_dim, paddin_idx=0)
    def forward(self, X, y=None, T_y=None, teacher_forcing=0.5, mask=None, starter = 1):
        h, h_last = self.encoder(X)
        logits, attn_weights = self.decoder(h, h_last, y, T_y, teacher_forcing, mask, starter)
        return logits, attn_weights

NameError: name 'nn' is not defined

In [122]:
from tqdm import tqdm

class MazeDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        super().__init__()
        d = {'<ADJLIST_START>':1,'<ADJLIST_END>':2, '<ORIGIN_START>':3,'<ORIGIN_END>':4,'<TARGET_START>':5,'<TARGET_END>':6, '<PATH_START>':7}
        mx = max(d.values())+1
        f = lambda x: x[0]*6+x[1]+mx
        V = f((5,5))+1
        def encode(df):
            y = []
            for i in tqdm(range(df.shape[0])):
                out_tokens = eval(df.iloc[i]['output_path'])
                in_tokens = eval(df.iloc[i]['input_sequence'])
                y_out = [1]
                y_in = []
                for token in in_tokens:
                    if token in ('<-->',';',): continue
                    y_in.append(f(eval(token)) if len(token) == 5 else d[token])
                for token in out_tokens:
                    if len(token) == 5:
                        p = eval(token)
                        y_out.append(p[0]*6+p[1]+3)
                    else: y_out.append(2)
                if len(y_in) < 129: y_in.extend([0]*(129-len(y_in))) #pad
                if len(y_out) < 38: y_out.extend([0]*(38-len(y_out))) 
                y.append((torch.LongTensor(y_in), torch.LongTensor(y_out))) 
            return y
        self.data = encode(df)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        return self.data[index]


In [123]:
df_train, df_val = train_test_split(df, train_size=0.9, random_state=42, stratify=df['maze_type'])
train_dataset = MazeDataset(df_train)
val_dataset = MazeDataset(df_val)
test_dataset = MazeDataset(df_test)

100%|██████████| 20000/20000 [00:08<00:00, 2409.70it/s]


In [127]:

def collate_fn(batch, pad_idx=0):
    inp = [item[0].unsqueeze(0) for item in batch]
    out = [item[1].unsqueeze(0) for item in batch]
    inp_seqs = torch.cat(inp, dim = 0)  # (batch, T_x)
    out_paths = torch.cat(out, dim = 0) # (batch, T_y)
    inp_mask = (inp_seqs != pad_idx)
    return {'inp': inp_seqs, 'out': out_paths, 'inp_mask': inp_mask}



In [130]:
from torch.amp import autocast, GradScaler
def train_epoch(model:seq2seqBahdanau, dataloader: torch.utils.data.DataLoader, optimizer, criterion, device, clip=1.0, teacher_forcing=0.5, scaler=None):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader):
        inp = batch['inp'].to(device)
        out = batch['out'].to(device)
        inp_mask = batch['inp_mask'].to(device)
        optimizer.zero_grad()
        with autocast(device_type=device.type):
            outputs, _ = model(inp,out, T_y=None,teacher_forcing=teacher_forcing, mask=inp_mask, starter=1)
            loss = criterion(outputs.view(-1, outputs.size(-1)), out[:,1:].contiguous().view(-1))
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

        total_loss += loss.item()*inp.size(0)
    return total_loss/len(dataloader.dataset)


In [ ]:
from torch.utils.data import DataLoader
from time import time
from sklearn.metrics import *
from collections import Counter

def evaluate(model, dataloader, criterion=None, device='cpu', pad_idx=0):
    """
    Returns: (avg_loss_per_token, seq_exact_accuracy, micro_f1)
    - avg_loss_per_token: cross-entropy averaged over non-pad tokens
    - seq_exact_accuracy: fraction of sequences with ALL tokens correct (ignoring pads)
    - micro_f1: micro-averaged F1 computed from token counts (ignoring pads)
    """
    model.eval()
    total_loss = 0.0        # sum of token losses (not averaged)
    total_tokens = 0       # number of non-pad tokens
    total_seq_wrongs = 0   # number of sequences that are NOT exactly matched
    total_tp = total_fp = total_fn = 0

    # We'll compute token loss by summing per-token loss using reduction='none' then masking
    with torch.no_grad():
        for batch in dataloader:
            inp = batch['inp'].to(device)          # (B, T_in)
            out = batch['out'].to(device)          # (B, T_out)  -- includes BOS maybe
            inp_mask = batch.get('inp_mask', None)
            if inp_mask is not None:
                inp_mask = inp_mask.to(device)

            # model should produce logits for targets excluding BOS, shape (B, T_y-1, V)
            logits, _ = model(inp, out, teacher_forcing=0, mask=inp_mask)  # logits: (B, T, V)
            preds = logits.argmax(dim=2)                                   # (B, T)

            targets = out[:, 1:].contiguous()    # (B, T)
            # mask to ignore PAD tokens in targets
            tgt_mask = (targets != pad_idx)      # bool (B, T)
            num_tokens = int(tgt_mask.sum().item())
            if num_tokens == 0:
                continue

            V = logits.size(-1)
            logits_flat = logits.view(-1, V)                 # (B*T, V)
            targets_flat = targets.view(-1)                  # (B*T,)

            # compute per-token loss and mask pads: safer than relying on criterion.reduction
            per_token_loss = F.cross_entropy(logits_flat, targets_flat, reduction='none')  # (B*T,)
            per_token_loss = per_token_loss.view(targets.size())                          # (B, T)
            token_loss_sum = (per_token_loss * tgt_mask).sum().item()

            total_loss += token_loss_sum
            total_tokens += num_tokens

            # sequence exact-match (ignoring pads)
            # wrong_seq_count = number of sequences where any non-pad token mismatches
            seq_wrong_mask = ((preds != targets) & tgt_mask).any(dim=1)   # (B,) bool
            total_seq_wrongs += int(seq_wrong_mask.sum().item())

            # micro F1 counts (exclude pads)
            for p_row, t_row, m_row in zip(preds, targets, tgt_mask):
                # filter out pad positions
                if not m_row.any():
                    continue
                p_list = p_row[m_row].tolist()
                t_list = t_row[m_row].tolist()
                pred_ctr = Counter(p_list)
                act_ctr = Counter(t_list)
                tp = sum((pred_ctr & act_ctr).values())
                fp = sum((pred_ctr - act_ctr).values())
                fn = sum((act_ctr - pred_ctr).values())
                total_tp += tp
                total_fp += fp
                total_fn += fn

    # Final metrics
    avg_loss = total_loss / total_tokens if total_tokens > 0 else 0.0
    seq_accuracy = 1.0 - (total_seq_wrongs / len(dataloader.dataset))

    if (total_tp + total_fp) > 0:
        p = total_tp / (total_tp + total_fp)
    else:
        p = 0.0
    if (total_tp + total_fn) > 0:
        r = total_tp / (total_tp + total_fn)
    else:
        r = 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

    return avg_loss, seq_accuracy, f1

def fit(model:seq2seqBahdanau, train_dataset, val_dataset, test_taset, pading_idx, batch_size=32, epochs=20,lr=1e-4,
        device=None,save_path="chkpt.pt",teacher_forcing=0.5):
    device=device or torch.device(0)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              collate_fn=lambda b: collate_fn(b, pad_idx=pading_idx))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                              collate_fn=lambda b: collate_fn(b, pad_idx=pading_idx))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                              collate_fn=lambda b: collate_fn(b, pad_idx=pading_idx))
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=pading_idx)
    scaler = GradScaler() if torch.cuda.is_available() else None
    model.to(device)
    best_valid_loss = float('inf')
    losses = []
    accs = []
    f1s = []
    for epoch in tqdm(range(1, epochs+1)):
        start = time()
        print(f"Starting Epoch {epoch}")
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device,
                                 clip=1.0, scaler=scaler, teacher_forcing=teacher_forcing)
        print(f"Evaluating. Train loss={train_loss}")
        train_loss, train_acc, train_f1 = evaluate(model, train_loader, criterion, device)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)
        test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
        losses.append((train_loss, val_loss, test_loss))
        accs.append((train_acc, val_acc, test_acc))
        f1s.append((train_f1, val_f1, test_f1))
        end = time()
        print(f"Epoch {epoch} | Time: {end-start:.1f}s \nTrain loss={train_loss:.4f} | Val loss={val_loss:.4f} | Test loss={test_loss:.4f}")
        print(f"Train acc={train_acc:.4f} | Val acc={val_acc:.4f} | Test acc={test_acc:.4f}")
        print(f"Train F1-score={train_f1:.4f} | Val F1-score={val_f1:.4f} | Test F1-score={test_f1:.4f}")
        if val_loss < best_valid_loss:
            best_valid_loss = val_loss
            # torch.save({
            #     'epoch': epoch,
            #     'model_state': model.state_dict(),
            #     'optimizer_state': optimizer.state_dict(),
            #     'val_loss': val_loss
            # }, "best_"+save_path)
            # print(f"model saved to best_{save_path}, epoch={epoch}")
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': val_loss
        }, str(epoch)+'_'+save_path)
        print(f"model saved to {epoch}_{save_path}")
    return {"model":model, "losses":losses, "accs": accs, "f1s":f1s}


In [ ]:
model = seq2seqBahdanau(45,128,39,128,512,512,512,39,)
model.state
out = fit(model, train_dataset, val_dataset, test_dataset, 0, device=device, save_path="checkpoint_.pt")

  0%|          | 0/20 [00:00<?, ?it/s]

Starting Epoch 1


100%|██████████| 2250/2250 [17:53<00:00,  2.10it/s]


Evaluating. Train loss=1.6002788033485413


  5%|▌         | 1/20 [26:02<8:14:48, 1562.55s/it]

...

Epoch 19 | Time: 1520.2s 
Train loss=0.5779 | Val loss=2.1662 | Test loss=2.1732
Train acc=0.0000 | Val acc=0.0000 | Test acc=0.0000
Train F1-score=0.0000 | Val F1-score=0.0000 | Test F1-score=0.0000
model saved to 19_checkpoint_no_clip.pt
Starting Epoch 20


100%|██████████| 2250/2250 [17:22<00:00,  2.16it/s]


Evaluating. Train loss=0.11630358224444919


100%|██████████| 20/20 [9:43:26<00:00, 1750.32s/it]


Epoch 20 | Time: 1516.6s 
Train loss=0.5116 | Val loss=2.2222 | Test loss=2.1605
Train acc=0.0000 | Val acc=0.0000 | Test acc=0.0000
Train F1-score=0.0000 | Val F1-score=0.0000 | Test F1-score=0.0000
model saved to 20_checkpoint_no_clip.pt


In [ ]:
'''
stop exec
fix code
run from checkpoint 4
run till checkpoint 20
get evaluations on test set for checkpoints 1 to 4
'''

In [113]:
losses, accs, f1s = out['losses'], out['accs'], out['f1s']
eps = list(range(5, len(losses)+5))
for metric, name in zip([losses, accs, f1s], ['loss', 'accuracy', 'F1-Score']):
    fig = plt.figure()
    for i, tp in enumerate(['train', 'val', 'test']):
        values = [val[i] for val in metric]
        plt.plot(eps, values, label = tp)
    plt.xlabel('epoch')
    plt.ylabel(name)
    plt.legend()
    plt.title(f"Epoch vs {name}")
    plt.savefig(f"Epoch-vs-{name}.png", dpi=300)
    plt.close(fig)

In [121]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                            collate_fn=lambda b: collate_fn(b, pad_idx=0))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                            collate_fn=lambda b: collate_fn(b, pad_idx=0))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,
                            collate_fn=lambda b: collate_fn(b, pad_idx=0))
criterion = nn.CrossEntropyLoss(ignore_index=0)
device = torch.device('mps')
losses, accs, f1s = [], [], []
for i in tqdm(range(1,21)):
    filename = str(i)+"_checkpoint.pt"
    old_model = seq2seqBahdanau(51,128,39,128,512,512,512,39,)
    state_dict = torch.load(filename)['model_state']
    old_model.load_state_dict(state_dict)
    old_model.to(device)
    train_loss, train_acc, train_f1 = evaluate(old_model, train_loader, criterion, device)
    val_loss, val_acc, val_f1 = evaluate(old_model, val_loader, criterion, device)
    test_loss, test_acc, test_f1 = evaluate(old_model, test_loader, criterion, device)
    losses.append((train_loss, val_loss, test_loss))
    accs.append((train_acc, val_acc, test_acc))
    f1s.append((train_f1, val_f1, test_f1))

eps = list(range(1, 21))
for metric, name in zip([losses, accs, f1s], ['loss', 'accuracy', 'F1-Score']):
    fig = plt.figure()
    for i, tp in enumerate(['train', 'val', 'test']):
        values = [val[i] for val in metric]
        plt.plot(eps, values, label = tp)
    plt.xlabel('epoch')
    plt.ylabel(name)
    plt.legend()
    plt.title(f"Epoch vs {name}")
    plt.savefig(f"Epoch-vs-{name}-w-clip-1.png", dpi=300)
    plt.close(fig)

 75%|███████▌  | 15/20 [3:31:28<1:10:29, 845.88s/it]


KeyboardInterrupt: 

In [81]:
accs = torch.load('checkpoints_clip_5;/METRICS.pt')['f1s']

In [79]:
accs

[(0.31270833333333337, 0.30674999999999997, 0.31045),
 (0.5121111111111112, 0.49175, 0.49245000000000005),
 (0.536513888888889, 0.509625, 0.5112),
 (0.5802916666666667, 0.54925, 0.5419499999999999),
 (0.6099861111111111, 0.569625, 0.5752999999999999),
 (0.6260972222222223, 0.570875, 0.5789),
 (0.6351249999999999, 0.5956250000000001, 0.5844),
 (0.6408888888888888, 0.586375, 0.5842499999999999),
 (0.6466666666666667, 0.591, 0.59285),
 (0.6543888888888889, 0.590125, 0.58745),
 (0.6685416666666666, 0.5958749999999999, 0.6019),
 (0.6768888888888889, 0.6036250000000001, 0.6011500000000001),
 (0.6833472222222222, 0.5994999999999999, 0.6059),
 (0.7029166666666666, 0.6152500000000001, 0.6148),
 (0.6940555555555555, 0.60075, 0.60025),
 (0.7015138888888889, 0.609, 0.605),
 (0.7273333333333334, 0.613875, 0.61785),
 (0.7347916666666667, 0.6154999999999999, 0.6173500000000001),
 (0.7362083333333334, 0.625625, 0.61875),
 (0.7448333333333333, 0.6191249999999999, 0.61185)]

In [194]:
[(j+1, i[0][0],i[0][1], i[0][2]) for j, i in enumerate(zip(accs,torch.load('checkpoints_clip_5_;_--/METRICS.pt')['accs'],torch.load('checkpoints_clip_5/METRICS.pt')['accs']))]

[(1, 0.31270833333333337, 0.30674999999999997, 0.31045),
 (2, 0.5121111111111112, 0.49175, 0.49245000000000005),
 (3, 0.536513888888889, 0.509625, 0.5112),
 (4, 0.5802916666666667, 0.54925, 0.5419499999999999),
 (5, 0.6099861111111111, 0.569625, 0.5752999999999999),
 (6, 0.6260972222222223, 0.570875, 0.5789),
 (7, 0.6351249999999999, 0.5956250000000001, 0.5844),
 (8, 0.6408888888888888, 0.586375, 0.5842499999999999),
 (9, 0.6466666666666667, 0.591, 0.59285),
 (10, 0.6543888888888889, 0.590125, 0.58745),
 (11, 0.6685416666666666, 0.5958749999999999, 0.6019),
 (12, 0.6768888888888889, 0.6036250000000001, 0.6011500000000001),
 (13, 0.6833472222222222, 0.5994999999999999, 0.6059),
 (14, 0.7029166666666666, 0.6152500000000001, 0.6148),
 (15, 0.6940555555555555, 0.60075, 0.60025),
 (16, 0.7015138888888889, 0.609, 0.605),
 (17, 0.7273333333333334, 0.613875, 0.61785),
 (18, 0.7347916666666667, 0.6154999999999999, 0.6173500000000001),
 (19, 0.7362083333333334, 0.625625, 0.61875),
 (20, 0.744833

In [195]:
[(j+1, i[1][0],i[1][1], i[1][2]) for j, i in enumerate(zip(accs,torch.load('checkpoints_clip_5_;_--/METRICS.pt')['accs'],torch.load('checkpoints_clip_5/METRICS.pt')['accs']))]

[(1, 0.22158333333333335, 0.21175, 0.21814999999999996),
 (2, 0.48887499999999995, 0.46399999999999997, 0.47450000000000003),
 (3, 0.5367777777777778, 0.518875, 0.5117499999999999),
 (4, 0.5695694444444445, 0.5489999999999999, 0.5445),
 (5, 0.6086944444444444, 0.5740000000000001, 0.5794),
 (6, 0.6292638888888888, 0.5916250000000001, 0.5886),
 (7, 0.6271111111111112, 0.589, 0.58815),
 (8, 0.648, 0.607, 0.60365),
 (9, 0.6475277777777777, 0.599, 0.5971),
 (10, 0.6605416666666667, 0.609625, 0.59935),
 (11, 0.6665138888888889, 0.6056250000000001, 0.6041000000000001),
 (12, 0.6758055555555555, 0.60975, 0.61075),
 (13, 0.6923472222222222, 0.623, 0.6163000000000001),
 (14, 0.6890972222222222, 0.61075, 0.6117),
 (15, 0.6996666666666667, 0.622375, 0.6188),
 (16, 0.7113194444444444, 0.62925, 0.6209),
 (17, 0.7233055555555555, 0.627875, 0.6234500000000001),
 (18, 0.729125, 0.6288750000000001, 0.62825),
 (19, 0.7419444444444445, 0.631875, 0.62905),
 (20, 0.741375, 0.63425, 0.623)]

In [5]:
import torch
[(j+1, i[1][0],i[1][1], i[1][2]) for j, i in enumerate(zip(torch.load('checkpoints_clip_5_;_--/METRICS.pt')['accs'],torch.load('checkpoints_clip_5;_--_do-10/METRICS.pt')['losses']))]

[(1, 2.5468273564479236, 2.5608169191839814, 2.5737969475721534),
 (2, 2.1615367597240076, 2.2495069826678202, 2.221029807048138),
 (3, 1.906753856450997, 1.951177602503262, 1.9673535090616119),
 (4, 1.7884288960231818, 1.8810272144032025, 1.8928701898913853),
 (5, 1.7001875527879047, 1.8074283088680443, 1.8483128535359383),
 (6, 1.6689980373059905, 1.76755552987684, 1.8003138941918666),
 (7, 1.5677401958262889, 1.7336682159103662, 1.7350986038094047),
 (8, 1.4746095651238735, 1.6290100466937865, 1.600447283824329),
 (9, 1.4290307493101022, 1.644297984081845, 1.592245477602519),
 (10, 1.3767480608500036, 1.4906723944755413, 1.528515109438907),
 (11, 1.318808183298148, 1.495416842339206, 1.5441284977261012),
 (12, 1.2418105385501879, 1.456527728334062, 1.4498428583454568),
 (13, 1.225079379105585, 1.4317011888219335, 1.4159285847037015),
 (14, 1.2202707692943289, 1.43066525657119, 1.4518521942156517),
 (15, 1.1246546186194113, 1.3125530419374682, 1.3764980507358506),
 (16, 1.09977349642

In [2]:
import torch

In [ ]:
def f(x):
    if x==0:
        return '0'
    if x==1:
        return '<PATH_START>'
    if x == 2:
        return '<PATH_END>'
    return f"({x//6},{x%6})"
def decode(preds: torch.Tensor, eos = 2, f=None):
    T_x = preds.size(1)
    eos_idx = (preds == eos).int().argmax(dim = 1)+1
    mask = torch.tensor([[True]*eos_idx[i].item()+[False]*(T_x-eos_idx[i].item()) for i in range(preds.size(0))], dtype=torch.bool)
    preds_pad = torch.multiply(preds, mask)
    if f is not None:
        seqs = [[f(j.item()) for j in preds_pad[i,:eos_idx[i].item()]] for i in range(preds.size(0))]
    else: seqs=None
    return preds_pad, seqs

In [50]:
def f(x):
    if x==0:
        return '0'
    if x==1:
        return '<PATH_START>'
    if x == 2:
        return '<PATH_END>'
    return f"({x//6},{x%6})"


In [75]:
preds = torch.tensor([[1,0,0],[0,0,0]])
a = (preds == 2).int().argmax(dim=1)

In [76]:
a

tensor([0, 0])

In [74]:
preds

tensor([[0, 0, 0],
        [0, 0, 0]])

In [54]:
preds[0]

tensor([1, 2, 3, 4])

In [63]:
decode(preds,f=f)

(tensor([[1, 2, 0, 0],
         [1, 4, 3, 2]]),
 [['<PATH_START>', '<PATH_END>'],
  ['<PATH_START>', '(0,4)', '(0,3)', '<PATH_END>']])

In [64]:
preds.any()

tensor(True)

In [ ]:
torch.randint(1,3,(1,),)


tensor([1])

In [77]:
accs

NameError: name 'accs' is not defined

In [87]:
type(eval('[1,2,3]'))

list

In [25]:
eval('[1+1]*8')

[2, 2, 2, 2, 2, 2, 2, 2]

In [88]:
p =(1,2)

In [91]:
a = torch.load('checkpoints_clip_5;_--_do-10/20_checkpoint_clip_5_--_do-10.pt')

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [6]:
import torch

In [18]:
birnn = torch.nn.RNN(4,10,2,bidirectional=True)

In [19]:
a = torch.tensor([[1,2,1,0],
                  [1,1,0,2]], dtype=torch.float32)

In [20]:
x = birnn(a)

In [23]:
x[0].size(),x[1].size()

(torch.Size([2, 20]), torch.Size([4, 10]))

In [24]:
a[[0,1]]

tensor([[1., 2., 1., 0.],
        [1., 1., 0., 2.]])